# Face Follower — Client V2 (MediaPipe)

Reads the MJPEG stream from the ESP32-CAM, detects faces with **MediaPipe Face Detection (BlazeFace)**, chooses the largest/closest detected face, decides a direction, and sends it to the ESP32 `/motor` endpoint.

This V2 keeps the original ESP32 HTTP API and motor logic. The main detection block is replaced with MediaPipe, with a small amount of temporal smoothing and command hysteresis to reduce LEFT/RIGHT jitter.

**Before running:** set `ESP32_IP` below to the IP printed by the ESP32 Serial Monitor.

**Important:** this notebook targets the legacy Python Face Detection API available in MediaPipe 0.10.21. That version supports Python 3.11 on Windows. It also uses `numpy<2` and `opencv-contrib-python<5` because MediaPipe 0.10.21 has a NumPy <2 compatibility requirement. After installing packages for the first time, restart the Jupyter kernel before continuing.


In [19]:
# ==================== INSTALL / IMPORT ====================
# Run this cell once in a fresh environment.
# After pip finishes, RESTART the Jupyter kernel, then run the import cell again.

%pip install "mediapipe==0.10.21" "numpy<2" "opencv-contrib-python<5" requests


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\erinx\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [20]:
import cv2
import mediapipe as mp
import numpy as np
import requests
import time
import threading
mp_face_detection = mp.solutions.face_detection

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("NumPy:", np.__version__)


OpenCV: 4.11.0
MediaPipe: 0.10.21
NumPy: 1.26.4


In [21]:
# ==================== CONFIG ====================
ESP32_IP = "10.158.116.178"          # <-- change to your ESP32's actual IP
STREAM_URL = f"http://{ESP32_IP}:81/stream"
MOTOR_URL = f"http://{ESP32_IP}/motor"

# Direction thresholds
CENTER_DEADZONE = 0.18       # +-20% of frame width around center = forward

TURN_PULSE_MS = 120
DIRECTION_CONFIRM_FRAMES = 3    
CLOSE_FACE_RATIO = 0.35      # face width > 35% of frame width = stop

# MediaPipe detection
MIN_DETECTION_CONFIDENCE = 0.55
MODEL_SELECTION = 0          # 0 = short range (best for a robot/person within ~2 m)
MAX_NUM_FACES = 5

# Temporal smoothing / command stability
FACE_SMOOTHING_ALPHA = 0.35  # lower = smoother, higher = more responsive
DIRECTION_CONFIRM_FRAMES = 2 # require same decision this many frames before changing command
NO_FACE_STOP_FRAMES = 5
COMMAND_COOLDOWN = 0.35      # minimum seconds between repeated HTTP commands

# Video / display
WINDOW_NAME = "Face Follower V2 - MediaPipe"
# ==================================================

mp_face_detection = mp.solutions.face_detection


### Fix applied
- Added the missing `run_face_follower()` function that Cell 8 was calling.
- Kept MediaPipe 0.10.21 and the existing motor API.
- Changed the old ESP32 IP to `YOUR_ESP32_IP`; replace it with the current IP from Serial Monitor.
- The loop now safely stops the motors when Q is pressed or the program exits.


In [22]:
# ==================== MEDIAPIPE DETECTOR ====================
# MediaPipe expects RGB input. It returns normalized bounding-box coordinates.

face_detector = mp_face_detection.FaceDetection(
    model_selection=MODEL_SELECTION,
    min_detection_confidence=MIN_DETECTION_CONFIDENCE,
)

print("MediaPipe Face Detection ready.")


MediaPipe Face Detection ready.


In [23]:
# ==================== MOTOR COMMAND SENDER ====================
# Same HTTP endpoint as the original notebook.
# Requests run on a background thread so a slow ESP32 response does not freeze video processing.

_last_sent_dir = None
_last_sent_time = 0.0
_command_lock = threading.Lock()


def send_direction(direction: str):
    """Send a direction to the ESP32, but avoid unnecessary HTTP spam."""
    global _last_sent_dir, _last_sent_time

    now = time.time()
    with _command_lock:
        if direction == _last_sent_dir and (now - _last_sent_time) < COMMAND_COOLDOWN:
            return
        _last_sent_dir = direction
        _last_sent_time = now

    def _worker():
        try:
            requests.get(
                MOTOR_URL,
                params={"dir": direction},
                timeout=0.8,
            )
        except requests.RequestException as e:
            print(f"[motor] request failed: {e}")

    threading.Thread(target=_worker, daemon=True).start()


In [24]:
# ==================== FACE DETECTION HELPERS ====================
def detection_to_box(detection, frame_w, frame_h):
    """Convert MediaPipe's normalized bounding box to (x, y, w, h) pixels."""
    bbox = detection.location_data.relative_bounding_box

    x = int(bbox.xmin * frame_w)
    y = int(bbox.ymin * frame_h)
    w = int(bbox.width * frame_w)
    h = int(bbox.height * frame_h)

    # Clamp the box to the actual frame. MediaPipe boxes can extend slightly outside it.
    x = max(0, min(x, frame_w - 1))
    y = max(0, min(y, frame_h - 1))
    w = max(1, min(w, frame_w - x))
    h = max(1, min(h, frame_h - y))

    return (x, y, w, h)


def detection_score(detection):
    """Return MediaPipe's first/primary detection confidence."""
    scores = detection.score
    return float(scores[0]) if scores else 0.0


def choose_largest_face(detections, frame_w, frame_h):
    """Choose the largest detected face, matching the original notebook's behavior."""
    candidates = []
    for detection in detections:
        box = detection_to_box(detection, frame_w, frame_h)
        x, y, w, h = box
        area = w * h
        candidates.append((area, detection_score(detection), box, detection))

    if not candidates:
        return None

    # Largest area first; confidence breaks ties.
    return max(candidates, key=lambda item: (item[0], item[1]))


def smooth_box(previous_box, new_box, alpha=FACE_SMOOTHING_ALPHA):
    """Exponential moving average for a steadier face box."""
    if previous_box is None:
        return new_box

    old = np.array(previous_box, dtype=np.float32)
    new = np.array(new_box, dtype=np.float32)
    smoothed = (1.0 - alpha) * old + alpha * new
    return tuple(int(v) for v in smoothed)


In [25]:
# ==================== DIRECTION LOGIC ====================
def decide_direction(face_box, frame_w, frame_h):
    x, y, w, h = face_box

    face_center_x = x + w / 2
    frame_center_x = frame_w / 2

    offset = (face_center_x - frame_center_x) / frame_w
    face_width_ratio = w / frame_w

    # Too close
    if face_width_ratio > CLOSE_FACE_RATIO:
        return "stop"

    # Turn only when face is outside the center zone
    if offset < -CENTER_DEADZONE:
        return "left"

    if offset > CENTER_DEADZONE:
        return "right"

    return "forward"


def stabilize_direction(candidate):
    """Require a candidate command to persist for a few frames before switching."""
    global _pending_direction, _pending_count, _stable_direction

    if candidate == _pending_direction:
        _pending_count += 1
    else:
        _pending_direction = candidate
        _pending_count = 1

    if _pending_count >= DIRECTION_CONFIRM_FRAMES:
        _stable_direction = candidate

    return _stable_direction


_pending_direction = None
_pending_count = 0
_stable_direction = "stop"


In [26]:
# ==================== MAIN FACE FOLLOWER LOOP ====================
def run_face_follower(show_window=True):
    """Read the ESP32-CAM MJPEG stream, detect the selected face,
    decide a direction, and send motor commands to the ESP32.
    Press Q to stop.
    """
    global _last_sent_dir, _last_sent_time
    global _pending_direction, _pending_count, _stable_direction

    cap = cv2.VideoCapture(STREAM_URL)

    if not cap.isOpened():
        print("ERROR: Cannot open ESP32-CAM stream.")
        print("Check ESP32_IP and open this URL in a browser first:")
        print(STREAM_URL)
        return

    # Keep the old command state clean when starting a new run.
    _last_sent_dir = None
    _last_sent_time = 0.0
    _pending_direction = None
    _pending_count = 0
    _stable_direction = "stop"

    previous_box = None
    no_face_count = 0

    print("Face follower started.")
    print("Stream:", STREAM_URL)
    print("Press Q to stop.")

    try:
        while True:
            ok, frame = cap.read()

            if not ok or frame is None:
                print("[video] failed to read frame")
                time.sleep(0.05)
                continue

            frame_h, frame_w = frame.shape[:2]

            # MediaPipe requires RGB input.
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_detector.process(rgb)

            selected = None

            if results.detections:
                selected = choose_largest_face(
                    results.detections,
                    frame_w,
                    frame_h
                )

            if selected is not None:
                _, confidence, box, _ = selected

                # Smooth the selected face position.
                previous_box = smooth_box(previous_box, box)

                direction_candidate = decide_direction(
                    previous_box,
                    frame_w,
                    frame_h
                )
                direction = stabilize_direction(direction_candidate)

                no_face_count = 0

                x, y, w, h = previous_box

                # Draw face box.
                cv2.rectangle(
                    frame,
                    (x, y),
                    (x + w, y + h),
                    (0, 255, 0),
                    2
                )

                cv2.putText(
                    frame,
                    f"Face {confidence:.2f}",
                    (x, max(25, y - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2
                )

                # Draw frame center and face center.
                frame_cx = frame_w // 2
                face_cx = x + w // 2

                cv2.line(
                    frame,
                    (frame_cx, 0),
                    (frame_cx, frame_h),
                    (255, 255, 0),
                    1
                )

                cv2.circle(
                    frame,
                    (face_cx, y + h // 2),
                    5,
                    (0, 0, 255),
                    -1
                )

                # Send the stabilized motor command.
                send_direction(direction)

                cv2.putText(
                    frame,
                    f"COMMAND: {direction.upper()}",
                    (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 255, 255),
                    2
                )

            else:
                no_face_count += 1

                # Do not immediately stop for one bad frame.
                if no_face_count >= NO_FACE_STOP_FRAMES:
                    previous_box = None
                    direction = stabilize_direction("stop")
                    send_direction("stop")

                cv2.putText(
                    frame,
                    "NO FACE",
                    (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 0, 255),
                    2
                )

            if show_window:
                cv2.imshow(WINDOW_NAME, frame)

                # Press Q to stop.
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break

    except KeyboardInterrupt:
        print("Stopped by user.")

    finally:
        # Always stop the robot when the program exits.
        send_direction("stop")
        time.sleep(0.15)

        cap.release()

        if show_window:
            cv2.destroyAllWindows()

        print("Face follower stopped safely.")


In [27]:
# ==================== RUN ====================
# Start the face follower.
# Press Q in the OpenCV window to stop.
run_face_follower(show_window=True)


Face follower started.
Stream: http://10.158.116.178:81/stream
Press Q to stop.
[motor] request failed: HTTPConnectionPool(host='10.158.116.178', port=80): Max retries exceeded with url: /motor?dir=stop (Caused by ConnectTimeoutError(<HTTPConnection(host='10.158.116.178', port=80) at 0x1aee3f603d0>, 'Connection to 10.158.116.178 timed out. (connect timeout=0.8)'))
[motor] request failed: HTTPConnectionPool(host='10.158.116.178', port=80): Max retries exceeded with url: /motor?dir=forward (Caused by ConnectTimeoutError(<HTTPConnection(host='10.158.116.178', port=80) at 0x1aee4cf83d0>, 'Connection to 10.158.116.178 timed out. (connect timeout=0.8)'))
[motor] request failed: HTTPConnectionPool(host='10.158.116.178', port=80): Max retries exceeded with url: /motor?dir=forward (Caused by ConnectTimeoutError(<HTTPConnection(host='10.158.116.178', port=80) at 0x1aee4c47f90>, 'Connection to 10.158.116.178 timed out. (connect timeout=0.8)'))
[motor] request failed: HTTPConnectionPool(host='10.1

## V2 notes

- The ESP32-CAM is unchanged at the protocol level: Python still sends `/motor?dir=forward|backward|left|right|stop`.
- MediaPipe returns normalized face bounding boxes, which are converted to the same `(x, y, w, h)` format used by the original `decide_direction()` logic.
- The largest face is still selected, preserving the original behavior when multiple faces are visible.
- A simple exponential moving average smooths the selected face box, and two-frame command confirmation reduces rapid LEFT/RIGHT command changes.
- The robot still performs full-speed pivot turns because the original Arduino wiring has ENA and ENB tied directly to 5V. PWM/proportional steering is a separate hardware/control upgrade.
- If the ESP32 stream itself is unstable, MediaPipe cannot fix that network/video problem; verify the stream URL first.
